In [4]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
import mlflow
import mlflow.xgboost
import json
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from mlflow.models.signature import infer_signature

df = pd.read_csv("dataset.csv")

In [7]:
def fill_missing_values(df, random_state=42):
    numeric_cols = df.select_dtypes(
        include=['number']
    ).columns
    categorical_cols = df.select_dtypes(
        include=['object', 'string', 'category']
    ).columns
    # Numeric imputation
    num_imputer = SimpleImputer(strategy='mean')
    df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])
    # One-hot encode categorical columns
    df = pd.get_dummies(
        df,
        columns=categorical_cols,
        dummy_na=True
    )
    # Final iterative imputation
    rf_imputer = IterativeImputer(
        estimator=RandomForestRegressor(
            random_state=random_state
        )
    )
    df = pd.DataFrame(
        rf_imputer.fit_transform(df),
        columns=df.columns
    )
    return df

# Call the function to fill missing values
df = fill_missing_values(df, random_state=42)

In [11]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

In [16]:
from imblearn.over_sampling import SMOTE

print('Before upsampling count of label 0 {}'.format(sum(y_train == 0)))
print('Before upsampling count of label 1 {}'.format(sum(y_train == 1)))

sm = SMOTE(sampling_strategy=1, random_state=1)

X_train_s, y_train_s = sm.fit_resample(X_train, y_train)

print('After upsampling count of label 0 {}'.format(sum(y_train_s == 0)))
print('After upsampling count of label 1 {}'.format(sum(y_train_s == 1)))

Before upsampling count of label 0 3743
Before upsampling count of label 1 761
After upsampling count of label 0 3743
After upsampling count of label 1 3743


In [23]:
columns = {'data_columns' : [col.lower() for col in X.columns]}

with open("models/columns.json","w") as f:
    f.write(json.dumps(columns))

In [ ]:
import subprocess

commit_hash = subprocess.check_output(

["git","rev-parse","HEAD"]

).decode().strip()

In [25]:
# Set experiment
mlflow.set_experiment("CustomerChurn")

with mlflow.start_run():

    # Initialize model
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss'
    )

    # Train model
    model.fit(X_train_s, y_train_s)

    signature = infer_signature(
    X_train_s,
    model.predict(X_train_s)
    )

    # Predictions
    y_pred = model.predict(X_test)

    # Probabilities for ROC-AUC
    y_pred_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred
    )
    recall = recall_score(
        y_test,
        y_pred
    )
    f1 = f1_score(
        y_test,
        y_pred
    )
    roc_auc = roc_auc_score(
        y_test,
        y_pred_prob
    )

    # Log hyperparameters
    mlflow.log_params(model.get_params())

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    from sklearn.metrics import confusion_matrix

    import matplotlib.pyplot as plt

    cm=confusion_matrix(
    y_test,
    y_pred
    )
    plt.savefig(
    "models/confusion_matrix.png"
    )

    # Log columns file
    mlflow.log_artifact(
        "models/columns.json"
    )

    # Log and register model
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        registered_model_name="CustomerChurnModel"
    )

    mlflow.set_tag(
        "owner",
        "Maitri"
    )

    mlflow.set_tag(
        "dataset_version",
        "v1"
    )
    mlflow.set_tag(
        "git_commit",
        commit_hash
    )

    mlflow.xgboost.log_model(
    xgb_model=model,
    artifact_path="model",
    signature=signature,
    input_example=X_train_s.iloc[:5],
    registered_model_name=
    "CustomerChurn"
    )

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC AUC  :", roc_auc)


2026/06/15 02:42:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/15 02:42:37 INFO mlflow.store.db.utils: Updating database tables
2026/06/15 02:42:45 INFO mlflow.tracking.fluent: Experiment with name 'CustomerChurn' does not exist. Creating a new experiment.
2026/06/15 02:42:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy : 0.9404973357015985
Precision: 0.879746835443038
Recall   : 0.7433155080213903
F1 Score : 0.8057971014492754
ROC AUC  : 0.9617353766949708


Successfully registered model 'CustomerChurnModel'.
Created version '1' of model 'CustomerChurnModel'.
